# MangaTR: أوراق مقارنة التنظيف

بياخد صفحات، يكشف النصوص، يعمل الماسك، وينضّف الصفحة بكذا طريقة، ويطلّع **ورقة لكل صفحة**: الأصل | الصناديق | الماسك | `opencv` | `aot` | `lama` | `cheap+lama` (الـ router المصمم)، وكل عمود عليه زمنه. وفي الآخر `timings.json` فيه الأزمنة وعدد الـ blocks اللي مرّ على كل طبقة.

**قبل ما تبدأ:** runtime **T4 GPU**، وسر `GITHUB_TOKEN` زي notebook الفحص (لو لسه ما أضفتوش شوف الـ notebook الأول).

الصفحات اللي هتتجرب: ارفعها في الخلية 4. جرّب 6 إلى 10 صفحات متنوعة (حبابة بيضاء، خلفية ملونة، نص فوق رسم، ويبتون طويل).

In [ ]:
# 1) الـ GPU
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "مفيش GPU")

In [ ]:
# 2) تحميل الكود وتثبيته
import os
import subprocess

from google.colab import userdata

BRANCH = "phase1/core-logic"
REPO_DIR = "/content/MangaTR"
PUBLIC_URL = "https://github.com/Nader-Geek/MangaTR.git"

if not os.path.isdir(REPO_DIR):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = (
        PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@") if token else PUBLIC_URL
    )
    done = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
        capture_output=True,
        text=True,
    )
    print((done.stdout + done.stderr).replace(token or "\0", "***")[-800:])
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL])
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"])
%pip install -q -e "/content/MangaTR[gpu]" pytest

In [ ]:
# 3) الموديلات (لو نزلت قبل كده هيتخطى التحميل)
%cd /content/MangaTR
M = "/content/models"
!python -m mangatr.gpu_check --models-dir $M --runs 3 --cpu-runs 0 --out /content/r.json | tail -3
!ls -la $M

In [ ]:
# 4) ارفع الصفحات (png / jpg / webp). بتتحفظ في /content/pages
import os

from google.colab import files

os.makedirs("/content/pages", exist_ok=True)
for name, data in files.upload().items():
    with open(f"/content/pages/{name}", "wb") as handle:
        handle.write(data)
print(os.listdir("/content/pages"))

In [ ]:
# 4-ب) بديل للخلية 4 لفصل كامل: من Google Drive (ملف zip فيه الصفحات أو فولدر صور)
# غيّر المسار تحت لمسار الملف عندك. لو رفعت الصفحات يدوياً في 4 تخطى الخلية دي.
import os
import zipfile

from google.colab import drive

SOURCE = "/content/drive/MyDrive/chapter.zip"  # أو فولدر صور
drive.mount("/content/drive")
os.makedirs("/content/pages", exist_ok=True)
if SOURCE.endswith(".zip"):
    with zipfile.ZipFile(SOURCE) as z:
        z.extractall("/content/pages")
else:
    !cp -r "$SOURCE"/. /content/pages/
print(len(os.listdir("/content/pages")), "عنصر في /content/pages")

In [ ]:
# 5) التشغيل
%cd /content/MangaTR
P, M, S = "/content/pages", "/content/models", "/content/sheets"
!python -m mangatr.compare --images $P --models-dir $M --out $S

In [ ]:
# 6) اعرض القصاصات المكبّرة (الأهم للحكم على الجودة). الأوراق الكاملة في sheets.zip
import glob

from IPython.display import Image, display

for path in sorted(glob.glob("/content/sheets/*_crops.jpg")):
    print(path)
    display(Image(path))

In [ ]:
# 7) ملخص صغير للإرسال (أرقام + أصعب القصاصات بحجم مصغّر)، بدل zip الأوراق الكاملة الكبير
import glob
import json
import os
import zipfile

from google.colab import files
from PIL import Image

S = "/content/sheets"
with open(f"{S}/timings.json", encoding="utf-8") as handle:
    timings = json.load(handle)
total, tiers, pages = {}, {}, 0
for name, rec in timings.items():
    if name.startswith("_"):
        continue
    pages += 1
    for variant, r in rec.items():
        if isinstance(r, dict) and "seconds" in r:
            total[variant] = total.get(variant, 0) + r["seconds"]
            for tier, n in r["blocks_by_tier"].items():
                tiers.setdefault(variant, {}).setdefault(tier, 0)
                tiers[variant][tier] += n
print("صفحات:", pages)
for variant in total:
    print(f"{variant:12s} {total[variant]:8.1f} ث  {tiers[variant]}")

os.makedirs("/content/summary", exist_ok=True)
small = []
for path in sorted(glob.glob(f"{S}/*_crops.jpg"))[:40]:
    im = Image.open(path)
    im.thumbnail((2400, 3000))
    out = f"/content/summary/{os.path.basename(path)}"
    im.save(out, quality=78)
    small.append(out)
with zipfile.ZipFile("/content/summary.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{S}/timings.json", "timings.json")
    for path in small:
        z.write(path, os.path.basename(path))
print("حجم summary.zip:", round(os.path.getsize("/content/summary.zip") / 1e6, 1), "MB")
files.download("/content/summary.zip")

## ابعتلي إيه

1. **`summary.zip`**: أرقام الأزمنة وعدد الـ blocks بكل طبقة، وقصاصات مكبّرة (مصغّرة الحجم) لأصعب الـ blocks. حجمه صغير يتبعت في الشات.
2. الأوراق الكاملة (`/content/sheets/*_sheet.jpg`) كبيرة جداً لفصل كامل: ما تبعتهاش كلها، اختار صفحتين أو تلاتة تحب أشوفهم.
3. لو خلية فشلت، آخر 15 سطر من خطأها (من غير التوكن).